In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_57_try_4.pickle

In [4]:
%%RecordEvent
%%time
### cell 58 ###

# Optimized cell for GPU via cudf.pandas

# Ensure question_of_interest_cell70 is defined
question_of_interest_cell70 = 'Do you use any of the following data storage products?'

# Step 1: Perform the column-name replacement on GPU
responses_df_2022_cell10.columns = (
    responses_df_2022_cell10.columns
      .to_series()
      .str.replace("(AWS/GCP/Azure customers only)", "for AWS GCP and Azure customers")
      .tolist()
)

# Step 2: Rewrite the subset function to use GPU string methods end-to-end
def grab_subset_of_data_70(original_df, question_of_interest):
    # GPU: turn the columns Index into a cudf.Series and mask by .str.contains
    col_series = original_df.columns.to_series()
    selected_cols = col_series[col_series.str.contains(question_of_interest)].index
    df = original_df[selected_cols]

    # GPU: split on '-' and strip leading whitespace to build new column names
    new_names = (
        df.columns
          .to_series()
          .str.split('-')
          .str.get(-1)
          .str.lstrip()
          .tolist()
    )
    df.columns = new_names

    # GPU: drop rows where all selected columns are null
    return df.dropna(how='all')

# Step 3: Apply the function and view the result
storage_df_2022 = grab_subset_of_data_70(
    responses_df_2022_cell10,
    question_of_interest_cell70
)
storage_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3735 entries, 8 to 23994
Data columns (total 8 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Microsoft Azure Blob Storage          615 non-null    object
 1   Microsoft Azure Files                 511 non-null    object
 2   Amazon Simple Storage Service (S3)    1624 non-null   object
 3   Amazon Elastic File System (EFS)      447 non-null    object
 4   Google Cloud Storage (GCS)            1288 non-null   object
 5   Google Cloud Filestore                481 non-null    object
 6   No / None                             771 non-null    object
 7   Other                                 74 non-null     object
dtypes: object(8)
memory usage: 262.6+ KB
CPU times: user 152 ms, sys: 0 ns, total: 152 ms
Wall time: 170 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_58_try_2.pickle